In [23]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from nltk.corpus import wordnet

# Pobranie pakietów NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/gabusia/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/gabusia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/gabusia/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/gabusia/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/gabusia/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [24]:
class InformationRetriever:
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        
    def preprocess(self, text):
        #Oczyszcza tekst, usuwa stop-words i dokonuje lematyzacji.
        words = word_tokenize(text.lower())
        cleaned_words = [
            self.lemmatizer.lemmatize(w) 
            for w in words 
            if w.isalnum() and w not in self.stop_words
        ]
        return cleaned_words # Zwracamy listę słów

    def ngram_similarity(self, words1, words2, n=2):
        #Zwraca podobieństwo Jaccarda dla n-gramów.
        if len(words1) < n or len(words2) < n:
            return 0.0
        
        ngrams1 = set(nltk.ngrams(words1, n))
        ngrams2 = set(nltk.ngrams(words2, n))
        
        intersection = ngrams1.intersection(ngrams2)
        union = ngrams1.union(ngrams2)
        
        return len(intersection) / len(union) if union else 0.0

    def wup_similarity_score(self, words1, words2):
        #Oblicza średnie podobieństwo Wu-Palmera
        if not words1 or not words2:
            return 0.0
            
        scores = []
        for w1 in words1:
            best_score = 0
            synsets1 = wordnet.synsets(w1)
            if not synsets1: continue
                
            for w2 in words2:
                synsets2 = wordnet.synsets(w2)
                if not synsets2: continue
                    
                # Szukamy najlepszego dopasowania dla danej pary słów
                for s1 in synsets1:
                    for s2 in synsets2:
                        score = s1.wup_similarity(s2)
                        if score and score > best_score:
                            best_score = score
            if best_score > 0:
                scores.append(best_score)
                
        # Zwracamy średnią z najlepszych dopasowań 
        return sum(scores) / len(scores) if scores else 0.0

    def find_best_sentence(self, context_text, question):
        sentences = sent_tokenize(context_text)

        # Preprocessing
        processed_sentences_lists = [self.preprocess(s) for s in sentences]
        processed_question_list = self.preprocess(question)
        
        # Łączymy w stringi dla TF-IDF
        processed_sentences_strings = [" ".join(s) for s in processed_sentences_lists]
        processed_question_string = " ".join(processed_question_list)

        # 1. TF-IDF + Cosine Similarity
        vectorizer = TfidfVectorizer()
        corpus = processed_sentences_strings + [processed_question_string]
        tfidf_matrix = vectorizer.fit_transform(corpus)
        cosine_sims = cosine_similarity(tfidf_matrix[-1], tfidf_matrix[:-1]).flatten()

        best_score = -1
        best_sentence = ""
        
        # Iterujemy po zdaniach i liczymy łączny wynik
        for i, sent_words in enumerate(processed_sentences_lists):
            # 2. Podobieństwo n-gramów
            ngram_score = self.ngram_similarity(processed_question_list, sent_words, n=2)
            
            # 3. Podobieństwo Wu-Palmera
            wup_score = self.wup_similarity_score(processed_question_list, sent_words)
            
            # 4. Kombinacja wyników 
            # Tu przykładowo 50% TF-IDF, 20% N-gramy, 30% Wu-Palmer
            final_score = (0.5 * cosine_sims[i]) + (0.2 * ngram_score) + (0.3 * wup_score)
            
            if final_score > best_score:
                best_score = final_score
                best_sentence = sentences[i]

        return best_sentence, best_score

<h4> Przykład użycia </h4>

In [30]:
nazwa_pliku = "gfr.txt" 
question = "What is the name of the youngest son of Vito?"

with open(nazwa_pliku, 'r', encoding='utf-8', errors='ignore') as file:
    text = file.read().replace('\n', ' ')

#Bierzemy tylko mały fragment tekstu do testów, bo inaczej przez algorytm Wu-Palmera program wykonuje się bardzo długo...
text = text[:5000] # Ograniczamy do pierwszych 5000 znaków (ok. 1-2 strony)
retriever = InformationRetriever() 
best_sentence, score = retriever.find_best_sentence(text, question)

print(f"\nPytanie: {question}")
print(f"Znalezione zdanie (Score: {score:.4f}):\n{best_sentence}")


Pytanie: What is the name of the youngest son of Vito?
Znalezione zdanie (Score: 0.3040):
He watched the  happy parents cluster around their darling sons.
